# Regularization — OLS vs. Ridge vs. Lasso vs. ElasticNet

**Objective:** compare plain linear regression against three regularized variants on the same dataset, to see how each shrinks coefficients and whether that improves generalization.
**Method:** OLS, RidgeCV, LassoCV, ElasticNetCV, each cross-validated to pick its own regularization strength.

**Data:** `sklearn.datasets.load_diabetes` — 442 patients, 10 baseline numeric features, target is disease progression.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score

PARAMS = {
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
    'alphas': np.logspace(-4, 2, 70),
    'l1_ratio': 0.5,
}

## STEP 01 - Define the Trainer

One class holding load/split/scale, fit-all-four-models, and evaluation, so each step below is a single method call.

In [2]:
class RegularizationTrainer:
    def __init__(self, params):
        self.params = params
        self.scaler = StandardScaler()
        self.models = {}
        self.results = None
        self.coef_df = None

    def prepare_data(self):
        diabetes = load_diabetes()
        X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
        y = diabetes.target

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.params['test_size'],
            random_state=self.params['random_state']
        )

        self.X_train_scaled = self.scaler.fit_transform(X_train)
        self.X_test_scaled = self.scaler.transform(X_test)
        self.y_train = y_train
        self.y_test = y_test
        self.feature_names = diabetes.feature_names

    def train_models(self):
        self.models['OLS'] = LinearRegression()

        self.models['Ridge'] = RidgeCV(
            alphas=self.params['alphas']
        )

        self.models['Lasso'] = LassoCV(
            alphas=self.params['alphas'],
            cv=self.params['cv_folds'],
            random_state=self.params['random_state']
        )

        self.models['ElasticNet'] = ElasticNetCV(
            alphas=self.params['alphas'],
            l1_ratio=self.params['l1_ratio'],
            cv=self.params['cv_folds'],
            random_state=self.params['random_state']
        )

        for name, model in self.models.items():
            model.fit(self.X_train_scaled, self.y_train)

    def evaluate(self):
        metrics = []
        coefficients = {'Fitur': self.feature_names}

        for name, model in self.models.items():
            y_pred = model.predict(self.X_test_scaled)
            score = r2_score(self.y_test, y_pred)
            best_alpha = getattr(model, 'alpha_', 'N/A')

            metrics.append({
                'Model': name,
                'R2 Score': score,
                'Best Alpha': best_alpha
            })
            coefficients[name] = model.coef_

        self.results = pd.DataFrame(metrics)
        self.coef_df = pd.DataFrame(coefficients)

    def display_results(self):
        print("\n" + "="*30)
        print("MODEL PERFORMANCE")
        print("="*30)
        print(self.results)

        print("\n" + "="*30)
        print("COEFFICIENT COMPARISON")
        print("="*30)
        print(self.coef_df)

## STEP 02 - Load, Split, Scale

In [3]:
trainer = RegularizationTrainer(PARAMS)
trainer.prepare_data()

## STEP 03 - Fit All Four Models

In [4]:
trainer.train_models()

## STEP 04 - Evaluate and Display

In [5]:
trainer.evaluate()
trainer.display_results()


MODEL PERFORMANCE
        Model  R2 Score Best Alpha
0         OLS  0.452603        N/A
1       Ridge  0.454366   1.221677
2       Lasso  0.471038   1.492496
3  ElasticNet  0.461141   0.201534

COEFFICIENT COMPARISON
  Fitur        OLS      Ridge      Lasso  ElasticNet
0   age   1.753758   1.816017   0.053450    1.890461
1   sex -11.511809 -11.435368  -8.248885   -9.882390
2   bmi  25.607121  25.749240  26.206473   24.328641
3    bp  16.828872  16.717025  15.176519   15.425506
4    s1 -44.448856 -33.094546  -5.161475   -5.587204
5    s2  24.640954  15.832268  -0.000000   -3.787195
6    s3   7.676978   2.675542 -11.171354   -8.942693
7    s4  13.138784  11.540881   0.000000    7.005844
8    s5  35.161195  30.766150  22.173946   19.192871
9    s6   2.351364   2.477470   1.859795    3.608240


## Conclusion

Lasso scores highest R² here and zeroes out the least-informative coefficients (feature selection as a side effect); Ridge and ElasticNet shrink coefficients without eliminating any; plain OLS overfits the noisiest features relative to all three.